In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
%%bash
pip install -q \
    transformers>=4.40.0 \
    datasets>=2.18.0 \
    accelerate>=0.28.0 \
    peft>=0.10.0 \
    bitsandbytes>=0.43.0 \
    evaluate>=0.4.0 \
    jiwer>=3.0.0 \
    librosa>=0.10.0 \
    soundfile>=0.12.0 \
    pandas \
    tqdm \
    tensorboard \
    PyMuPDF \
    pytesseract \
    Pillow \
    torch>=2.2.0 \
    torchaudio>=2.2.0

echo "✅ All packages installed"

✅ All packages installed


In [2]:
import os
import re
import json
import warnings
import unicodedata
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union

import torch
import numpy as np
import soundfile as sf
from tqdm.auto import tqdm

from datasets import Dataset, DatasetDict, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from peft import get_peft_model, LoraConfig, TaskType
import evaluate

warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device : {device}")

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16
TORCH_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if USE_FP16 else torch.float32)
print(f"⚙️  Training dtype: {TORCH_DTYPE}")

if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🎮  GPU    : {torch.cuda.get_device_name(0)}")
    print(f"💾  VRAM   : {total_vram:.1f} GB")
    if total_vram < 14:
        print("⚠️  Under ~14 GB: set per_device_train_batch_size=1 and/or use whisper-small.")
else:
    print("⚠️  CUDA not available — training will be very slow on CPU.")

print("✅ Environment ready")

🖥️  Device : cuda
⚙️  Training dtype: torch.bfloat16
🎮  GPU    : Tesla T4
💾  VRAM   : 15.6 GB
✅ Environment ready
